In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, coalesce

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    # Latest date where the derivative settlement table exists (it can lag landed/
    # curated by a few days). Pin ALL reads to this one date so the landed
    # vdebtorsrelations join is date-consistent with dsettlementexposure.
    run_date = str(
        spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

print(f"env:       {env}")
print(f"run_date:  {run_date}")

# dtransfer_SettlementDebtor (DNA26-1804)

New delivery table at **settlement × debtor** grain, reproducing IT's profits
settlement-debtor perimeter query. One row per (settlement, debtor), binding each
settlement to its applicant and its involved debtors (relation + VAT).

## Sources

| Field group | Source | Notes |
| --- | --- | --- |
| Perimeter + keys + case/phase fields + applicant | `{env}-cdh.derivative.dsettlementexposure` | Filter `isPromiseActive = 1` AND `SourceSystem = 'profits'`. Carries SettlementKey, ExposureKey, AgreementKey, CDI (= applicant), SettlementStatus (interim — derivative status, not IT CASE_SETTL_STATUS), CreatedDate, PortfolioKey, and the R0.A fields PromiseCaseID / PromiseCasePurpose / PromiseCaseDescription / PromiseCaseArrivalDate / SettlementPhase / SettlementJustification. |
| Debtor fan-out (DebtorCode / DebtorRelation / DebtorVatNumber) | `{env}-cdh.landed.profits_vdebtorsrelations` | **Direct landed read** — a deliberate, scoped exception (debtor relations not tied to settlements are of no interest to curated/derivative; IT query is profits-only). Loan rows join on `ACCOUNT_NUMBER`, agreement rows on `CONTRACT_NUMBER` (keys match the settlement keys directly). |
| Banking exclusion | `PortfolioKey`/PortfolioId on the derivative rows | Always exclude 31/32/60/92/87. |

## Grain / limitations

- `SELECT DISTINCT` on the IT column set (the raw fan inflates because a contract
  recurs across cases/promises and VDebtorsRelations carries repeat rows).
- **Profits-only.** The ~1,559 still-active QC→QCA migrated 1→N settlements (qca) are
  NOT covered (no profits clc case / vdebtors rows), mirroring IT's query.
- Applicant = `dsettlementexposure.CDI` (= IT OtherId type-13 ID_NO, 99.94%).
- `HasActiveSettlementFlag` = 1 on the debtor row whose CDI = the settlement's
  applicant CDI and that has an active primary settlement.

## Output

- `{env}-cdh-delivery.it.dtransfer_settlementdebtor` — daily overwrite.

In [0]:
# Banking portfolios ALWAYS excluded (unconditional, DNA26-1804).
BANKING_PORTFOLIO_IDS = [31, 32, 60, 92, 87]

# ---- Settlement perimeter from the derivative (profits, isPromiseActive) ----
# One row per (SettlementKey, ExposureKey) in the derivative; we need one row per
# settlement carrying its case fields + applicant + the loan/agreement key, plus
# the case level. dsettlementexposure fills BOTH keys, so case level comes from
# curated primary.settlementexposure (ExposureKey NULL + AgreementKey present =
# agreement).
ds = (
    spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("SourceSystem") == "profits")
        & (col("isPromiseActive") == 1)
    )
)

# Case level from curated (natural loan/agreement split).
curated_level = (
    spark.table(f"`{env}-cdh`.primary.settlementexposure")
    .filter((col("BusinessDate") == run_date_lit) & (col("SourceSystem") == "profits"))
    .select(
        col("SettlementKey"),
        when(col("ExposureKey").isNull() & col("AgreementKey").isNotNull(), lit("agreement"))
        .otherwise(lit("loan")).alias("_case_level"),
    )
    .distinct()
)

# Portfolio lookup for banking exclusion.
portfolio = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .filter(col("BusinessDate") == run_date_lit)
    .select(col("PortfolioId"), col("PortfolioName").alias("_pf_name"))
    .distinct()
)

# One row per settlement: pick the distinct settlement-level attributes.
# dsettlementexposure repeats these across exposure rows; dedup to settlement grain.
settlement = (
    ds.select(
        col("SettlementKey"),
        col("ExposureKey"),
        col("AgreementKey"),
        col("CDI").alias("ApplicantCDI"),
        col("SettlementStatus"),
        col("CreatedDate"),
        col("PortfolioName"),
        col("PromiseCaseID"),
        col("PromiseCasePurpose"),
        col("PromiseCaseDescription"),
        col("PromiseCaseArrivalDate"),
        col("SettlementPhase"),
        col("SettlementJustification"),
    )
    .distinct()
    .join(curated_level, on="SettlementKey", how="left")
    .join(portfolio, col("PortfolioName") == col("_pf_name"), how="left")
    .filter(~col("PortfolioId").isin(*BANKING_PORTFOLIO_IDS))
)

# Loan vs agreement natural key for the debtor join + the AccountNumber/AgreementNumber split.
settlement = settlement.select(
    col("SettlementKey"),
    col("_case_level"),
    when(col("_case_level") == "agreement", lit(None)).otherwise(col("ExposureKey")).alias("AccountNumber"),
    when(col("_case_level") == "agreement", col("AgreementKey")).otherwise(lit(None)).alias("AgreementNumber"),
    col("ApplicantCDI"),
    col("SettlementStatus"),
    col("CreatedDate"),
    col("PromiseCaseID"),
    col("PromiseCasePurpose"),
    col("PromiseCaseDescription"),
    col("PromiseCaseArrivalDate"),
    col("SettlementPhase"),
    col("SettlementJustification"),
)

print(f"perimeter settlements (profits, non-banking): {settlement.select('SettlementKey').distinct().count():,}")

In [0]:
# ---- Debtor relations (DIRECT landed read — scoped exception, DNA26-1804) ----
# Loan rows join on ACCOUNT_NUMBER, agreement rows on CONTRACT_NUMBER. Keys match
# the settlement keys directly (no prefix-strip).
vd = (
    spark.table(f"`{env}-cdh`.landed.profits_vdebtorsrelations")
    .filter(col("BusinessDate") == run_date_lit)
    .select(
        col("ACCOUNT_NUMBER"),
        col("CONTRACT_NUMBER"),
        col("CDI").alias("DebtorCode"),
        col("RELATION_CODE").alias("DebtorRelation"),
        col("AFM_NO").alias("DebtorVatNumber"),
    )
)

vd_loan = vd.filter(col("ACCOUNT_NUMBER").isNotNull()).select(
    col("ACCOUNT_NUMBER").alias("_k"), "DebtorCode", "DebtorRelation", "DebtorVatNumber"
)
vd_con = vd.filter(col("CONTRACT_NUMBER").isNotNull()).select(
    col("CONTRACT_NUMBER").alias("_k"), "DebtorCode", "DebtorRelation", "DebtorVatNumber"
)

loan_rows = (
    settlement.filter(col("_case_level") == "loan")
    .join(vd_loan, col("AccountNumber") == col("_k"), how="left")
    .drop("_k")
)
agr_rows = (
    settlement.filter(col("_case_level") == "agreement")
    .join(vd_con, col("AgreementNumber") == col("_k"), how="left")
    .drop("_k")
)

fanned = loan_rows.unionByName(agr_rows)

# ---- HasActiveSettlementFlag ----
# 1 where the debtor is the applicant (DebtorCode = ApplicantCDI) AND the settlement
# is an active primary settlement for that key+CDI (dsettlementexposure).
active_primary = (
    spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("SourceSystem") == "profits")
        & (col("IsActiveFlag") == 1)
        & (col("isPrimarySettlementFlag") == True)
    )
    .select(col("SettlementKey").alias("_ap_sk"), col("CDI").alias("_ap_cdi"))
    .distinct()
)

fanned = (
    fanned.join(
        active_primary,
        (col("SettlementKey") == col("_ap_sk")) & (col("DebtorCode") == col("_ap_cdi")),
        how="left",
    )
    .withColumn(
        "HasActiveSettlementFlag",
        when(col("_ap_sk").isNotNull(), lit(1)).otherwise(lit(0)),
    )
    .drop("_ap_sk", "_ap_cdi")
)

# ---- Final columns (IT query order) ----
settlementdebtor_df = (
    fanned.select(
        run_date_lit.alias("ReferenceDate"),
        col("PromiseCaseID").alias("CaseCode"),
        col("PromiseCasePurpose").alias("CasePurpose"),
        when(col("_case_level") == "agreement", lit("\u03a3\u03b5 \u03b5\u03c0\u03af\u03c0\u03b5\u03b4\u03bf \u03c3\u03cd\u03bc\u03b2\u03b1\u03c3\u03b7\u03c2"))
        .otherwise(lit("\u03a3\u03b5 \u03b5\u03c0\u03af\u03c0\u03b5\u03b4\u03bf \u0394\u03b1\u03bd\u03b5\u03af\u03bf\u03c5")).alias("CaseLevel"),
        col("PromiseCaseDescription").alias("CaseDescription"),
        col("PromiseCaseArrivalDate").alias("CaseStartDate"),
        col("SettlementKey").cast("string").alias("SettlementCode"),
        col("CreatedDate").cast("date").alias("SettlementCreateDate"),
        col("SettlementStatus"),
        col("SettlementPhase"),
        col("SettlementJustification"),
        col("AccountNumber"),
        col("AgreementNumber"),
        col("ApplicantCDI").alias("CustomerCode"),
        col("DebtorCode"),
        col("DebtorRelation"),
        col("DebtorVatNumber"),
        col("HasActiveSettlementFlag"),
    )
    # IT uses SELECT DISTINCT: the raw fan inflates via contract-recurrence / vdebtor repeats.
    .distinct()
)

sd_count = settlementdebtor_df.count()
print(f"dtransfer_settlementdebtor rows: {sd_count:,}")

# ---- Pre-write guards ----
if sd_count == 0:
    raise ValueError("dtransfer_settlementdebtor produced 0 rows — aborting (empty-result guard)")

target_settlementdebtor = f"`{env}-cdh-delivery`.it.dtransfer_settlementdebtor"
if spark.catalog.tableExists(target_settlementdebtor):
    prev_count = spark.table(target_settlementdebtor).count()
    if prev_count > 0 and sd_count < prev_count * 0.75:
        raise ValueError(
            f"dtransfer_settlementdebtor row-count floor breached: {sd_count:,} < 75% of "
            f"previous {prev_count:,}. Investigate before overwriting."
        )

In [0]:
(
    settlementdebtor_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_settlementdebtor)
)

print(f"Written to {target_settlementdebtor}")